# Notebook 01 — Capstone Design Principles

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 1 of 12  
**Time estimate:** 60 minutes

> A portfolio project differs from a homework exercise in one way: it is designed
> to be read by someone who didn't assign it, who has no obligation to be
> generous, and who is comparing it against hundreds of other candidates.
> This notebook defines the standard.

---
## Step 1 — What Makes a Portfolio-Quality Project

| Criterion | Homework exercise | Portfolio project |
|-----------|------------------|------------------|
| **Biological question** | Assigned | Self-chosen, real, defensible |
| **Data** | Provided, clean | Obtained, documented, provenance clear |
| **Code** | Runs locally | Runs from a clean clone, one command |
| **Figures** | Draft quality | Publication quality (Module 18 NB07 standard) |
| **Writing** | Informal | Abstract + methods section (Module 18 NB02/04 standard) |
| **Testing** | None | At least smoke tests (assert outputs have expected shape/range) |
| **Documentation** | Inline comments | README, docstrings, ENVIRONMENT.md |

A project that scores ≥5/7 on these criteria is portfolio-quality.
Below 4 is homework. The capstones below target 7/7.

---
## Step 2 — The Three Capstone Projects

| Project | Sub-field | Key methods | Track A value | Track B value |
|---------|-----------|-------------|--------------|---------------|
| **CP01** RNA-seq DE | Genomics/transcriptomics | NB counts, normalization, Wald test, FDR | ★★★★ (NGS pipeline) | ★★★ (bioinformatics background) |
| **CP02** TF binding | ML + genomics | k-mer features, SVM, CNN, 5-fold CV | ★★★★ (ML in genomics) | ★★★★ (ML+bio narrative) |
| **CP03** Epidemic models | Simulation/systems | ODE, network SIR, ABM, MCMC | ★★★ (quantitative biology) | ★★★ (computational modelling) |

Together they cover 3 of the 5 major computational biology sub-fields,
demonstrate both analysis and methods skills, and produce 3 independent
written artifacts — enough for a coherent PhD application narrative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List

@dataclass
class CapstoneCriteria:
    biological_question: float = 0.0   # 0–1
    data_documented:     float = 0.0
    reproducible:        float = 0.0
    pub_figures:         float = 0.0
    writing_quality:     float = 0.0
    testing:             float = 0.0
    documentation:       float = 0.0

    def score(self): return np.mean([self.biological_question, self.data_documented,
                                      self.reproducible, self.pub_figures,
                                      self.writing_quality, self.testing, self.documentation])

@dataclass
class CapstonePlanner:
    projects: List[dict] = field(default_factory=list)

    def add(self, name, sub_field, track_a, track_b, criteria: CapstoneCriteria):
        self.projects.append({'name': name, 'sub_field': sub_field,
                               'track_a': track_a, 'track_b': track_b,
                               'criteria': criteria, 'score': criteria.score()})

    def coverage(self):
        sub_fields = [p['sub_field'] for p in self.projects]
        return set(sub_fields)

    def readiness(self):
        if not self.projects: return 0.0
        return np.mean([p['score'] for p in self.projects])

# Define the three capstones
planner = CapstonePlanner()
planner.add('CP01 RNA-seq DE', 'genomics',
            track_a=0.9, track_b=0.7,
            criteria=CapstoneCriteria(0.9, 0.85, 0.85, 0.9, 0.8, 0.7, 0.85))
planner.add('CP02 TF Binding', 'ML+genomics',
            track_a=0.9, track_b=0.95,
            criteria=CapstoneCriteria(0.9, 0.9, 0.9, 0.95, 0.85, 0.8, 0.9))
planner.add('CP03 Epidemic', 'simulation',
            track_a=0.7, track_b=0.8,
            criteria=CapstoneCriteria(0.85, 0.85, 0.9, 0.85, 0.8, 0.7, 0.8))

for p in planner.projects:
    print(f"{p['name']:25} score={p['score']:.2f}  TrackA={p['track_a']:.1f}  TrackB={p['track_b']:.1f}")
print(f"\nCoverage: {planner.coverage()}")
print(f"Mean readiness: {planner.readiness():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Portfolio coverage radar
ax = axes[0]
criteria_labels = ['Biological\nquestion', 'Data\ndocumented', 'Reproducible',
                    'Pub\nfigures', 'Writing\nquality', 'Testing', 'Documentation']
n_axes = len(criteria_labels)
angles = np.linspace(0, 2*np.pi, n_axes, endpoint=False).tolist()
angles += angles[:1]

for p, color in zip(planner.projects, ['#4e79a7', '#f28e2b', '#59a14f']):
    c = p['criteria']
    vals = [c.biological_question, c.data_documented, c.reproducible,
              c.pub_figures, c.writing_quality, c.testing, c.documentation]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, lw=2, label=p['name'])
    ax.fill(angles, vals, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria_labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_title('A. Capstone quality radar\n(7 portfolio criteria)')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)

# Panel B: Track A/B relevance
ax = axes[1]
names = [p['name'] for p in planner.projects]
track_a = [p['track_a'] for p in planner.projects]
track_b = [p['track_b'] for p in planner.projects]
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, track_a, w, label='Track A (India)', color='#4e79a7', edgecolor='black', alpha=0.8)
ax.bar(x + w/2, track_b, w, label='Track B (EU PhD)', color='#f28e2b', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8)
ax.set_ylim(0, 1.1); ax.set_ylabel('Relevance score (0–1)')
ax.set_title('B. Track A vs B relevance\nper capstone project')
ax.legend(fontsize=9)

# Panel C: 12-month timeline
ax = axes[2]
ax.axis('off')
months = [
    (1,  2,  'Modules 01–04\n(Python, Math, Stats, LinAlg)', '#d9e8f5'),
    (3,  4,  'Modules 05–10\n(Biology, Genomics, NGS, Omics)', '#d9f0e8'),
    (5,  6,  'Modules 11–13\n(Structure, Systems, ML)\n+ Track A checkpoint', '#fff3cc'),
    (7,  9,  'Modules 14–17\n(DL, Simulation, RSE, HPC)', '#fce4d6'),
    (10, 11, 'Module 18–19\n(Writing, Paper Reading)', '#e8d9f5'),
    (12, 12, 'Module 20\nCapstone + PhD Apps', '#f5d9d9'),
]
for (m1, m2, label, color) in months:
    y = 1.0 - (m1 - 1) / 12
    h = (m2 - m1 + 1) / 12 * 0.9
    ax.add_patch(mpatches.FancyBboxPatch((0.05, y - h), 0.9, h,
                                           boxstyle='round,pad=0.01', facecolor=color,
                                           edgecolor='grey', transform=ax.transAxes))
    ax.text(0.5, y - h/2, label, transform=ax.transAxes, ha='center', va='center', fontsize=7.5)
ax.set_title('C. 12-month programme timeline')

plt.suptitle('Module 20 NB01: Capstone Design Principles', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('capstone_design.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

Define a fourth hypothetical capstone project (not one of the three above) in a
sub-field you find genuinely interesting. Instantiate it with `CapstonePlanner`,
score it, and verify it does not duplicate any of the three existing sub-fields.

---
## Step 12 — Reflection

> *[In one paragraph: which of the three capstone projects is most aligned with
> your genuine biological interest? Why? How does it connect to the target
> PhD supervisors you identified in Module 18 NB10?]*

---
*Next: `02_cp01_data_generation_qc.ipynb`*